# Fine-tuning Mistral-7B for receipt extraction (QLoRA)

This notebook fine-tunes **Mistral-7B-Instruct** to turn messy receipt text into
structured JSON, using **QLoRA** (4-bit base + LoRA adapters) so the whole thing
fits on a free Colab **T4 (15 GB)**.

**Before you run:** `Runtime -> Change runtime type -> T4 GPU`. Then run the cells
top to bottom. A full run on ~2,000 examples is roughly 30-45 min on a T4.

The pipeline: load the dataset built by `src/generate_dataset.py` -> 4-bit load the
base model -> attach LoRA -> train on *completion-only* loss -> **evaluate fine-tuned
vs. base** with `eval/evaluate.py` -> merge + save for GGUF conversion.

## 1. Install dependencies

Pinned to recent stable lines. If an import breaks on a future release, that's the first place to look.

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "accelerate>=0.33" \
    "bitsandbytes>=0.43" "datasets>=2.20" "pydantic>=2.6"

# Mistral-7B-Instruct is a gated model: accept its license once at
# https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3 then paste a
# read token below so the download is authorised.
from huggingface_hub import login
login()

## 2. Get the project code + generate the dataset

We pull the repo so the notebook imports the **exact same** `schema`, `prompt`,
and `evaluate` modules the rest of the project uses. That shared prompt is what keeps
training and inference from drifting apart.

Replace the URL with your repo once it's pushed.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Ogirala-Uday-Venkat-Rahul/receipt-extractor.git"
if not os.path.exists("receipt-extractor"):
    !git clone $REPO_URL
sys.path.append(os.path.abspath("receipt-extractor"))
%cd receipt-extractor

# Reproduce the dataset locally (deterministic given the seed).
!python -m src.generate_dataset --train 2000 --val 200 --test 200 --seed 7

## 3. Load and format the data

Each training row is `prompt + target + </s>`, built through `src.prompt` so it
matches inference byte-for-byte. We keep the raw `text`/`target` around too - the
evaluator needs them later.

In [ ]:
import json
from src.prompt import build_prompt, build_training_example

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding='utf-8')]

train_rows = load_jsonl('data/train.jsonl')
val_rows   = load_jsonl('data/val.jsonl')

print('train:', len(train_rows), '| val:', len(val_rows))
print('\n--- one formatted training example ---\n')
print(build_training_example(train_rows[0]['text'], train_rows[0]['target'])[:600])

## 4. Load Mistral-7B in 4-bit

The **QLoRA** recipe: the frozen base weights live in 4-bit NF4 (that's what makes
7B fit in 15 GB), while compute runs in bf16. We only ever train the small LoRA
adapters on top - the 7B itself never moves.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False  # required with gradient checkpointing

## 5. Tokenize with completion-only labels

The key detail. We compute loss **only on the JSON answer**, not on the instruction
or the receipt text. We tokenize the prompt separately, then mask those positions in
`labels` with `-100` (which the loss ignores). Without this the model wastes capacity
learning to echo the prompt.

In [ ]:
from datasets import Dataset

MAX_LEN = 1024

def tokenize(row):
    prompt = build_prompt(row['text'])
    full   = build_training_example(row['text'], row['target'])
    # add_special_tokens=False: our prompt already carries <s>/</s> literally.
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    full_ids   = tokenizer(full, add_special_tokens=False,
                           truncation=True, max_length=MAX_LEN)['input_ids']
    labels = list(full_ids)
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100  # mask the prompt; train only on the answer
    return {'input_ids': full_ids, 'attention_mask': [1]*len(full_ids), 'labels': labels}

train_ds = Dataset.from_list(train_rows).map(tokenize, remove_columns=['text','target'])
val_ds   = Dataset.from_list(val_rows).map(tokenize, remove_columns=['text','target'])

## 6. Attach LoRA adapters

LoRA freezes the base and injects small trainable rank-`r` matrices into the
attention and MLP projections. With `r=16` we train on the order of 0.1-0.2% of the
parameters - that's why a 7B model fine-tunes on a free GPU.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 7. Train

A plain `Trainer` - nothing hidden. `paged_adamw_8bit` keeps the optimizer state
small, gradient accumulation gives an effective batch of 16 on a T4, and cosine decay
with a short warmup is the standard LoRA schedule. 2-3 epochs is plenty for a narrow
extraction task; watch val loss and stop when it flattens.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

args = TrainingArguments(
    output_dir='out',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collator)
trainer.train()

## 8. A generation helper

Greedy decode, then slice off everything up to and including `[/INST]` so we're left
with just the model's JSON. This same function drives the evaluation.

In [ ]:
from src.prompt import build_prompt

def make_predictor(m):
    m.eval()
    def predict(receipt_text: str) -> str:
        prompt = build_prompt(receipt_text)
        ids = tokenizer(prompt, add_special_tokens=False, return_tensors='pt').to(m.device)
        with torch.no_grad():
            out = m.generate(**ids, max_new_tokens=512, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
        return text.split('[/INST]')[-1].strip()
    return predict

## 9. The money shot - fine-tuned vs. base

This is the result the project exists to produce. We score both models on the
held-out **test** set with identical metrics. Disabling the LoRA adapters gives us the
base model for free - same weights, same 4-bit load, adapters off - so the comparison
is genuinely apples-to-apples.

In [ ]:
from eval.evaluate import load_examples, evaluate, format_report

test_examples = load_examples('data/test.jsonl')
predict = make_predictor(model)

# Base model = same model with the LoRA adapters switched off.
with model.disable_adapter():
    base_report = evaluate(predict, test_examples)

ft_report = evaluate(predict, test_examples)

print(format_report({'base (Mistral-7B)': base_report, 'fine-tuned': ft_report}))

## 10. Merge + save for serving

For CPU serving on a free HF Space we can't use the 4-bit + LoRA stack (bitsandbytes
needs a GPU). So we **merge** the adapters into fp16 weights, then convert to **GGUF**
and quantize with llama.cpp for fast CPU inference.

The merge reloads the base in fp16 (not 4-bit) so the merged weights are clean.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

trainer.model.save_pretrained('adapters')  # small - keep these too

base_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map='cpu')
merged = PeftModel.from_pretrained(base_fp16, 'adapters')
merged = merged.merge_and_unload()
merged.save_pretrained('merged')
tokenizer.save_pretrained('merged')
print('merged model written to ./merged')

## 11. Convert to quantized GGUF

llama.cpp ships a converter. `Q4_K_M` is the sweet spot - ~4 GB, minimal quality
loss, comfortably fast on a Space's CPU. Download the resulting `.gguf` and point the
serving app's `MODEL_PATH` at it.

*(This step is heavy - it materialises the fp16 model. If Colab is tight on disk, do
the conversion in a fresh runtime that only loads `./merged`.)*

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py merged --outfile receipt-extractor-f16.gguf --outtype f16
!cd llama.cpp && cmake -B build && cmake --build build --config Release -j
!./llama.cpp/build/bin/llama-quantize receipt-extractor-f16.gguf receipt-extractor-q4.gguf Q4_K_M

from google.colab import files
files.download('receipt-extractor-q4.gguf')